# 第二十课:Mini Transformer(总装)

讲义:[20-mini-gpt.md](20-mini-gpt.md)。本 notebook 就是讲义第 2 节代码的执行现场,输出与讲义第 3 节逐字对应。

In [ ]:
import numpy as np, math

def softmax(x, axis=-1):
    m = np.max(x, axis=axis, keepdims=True)
    e = np.exp(x - m)
    return e / e.sum(axis=axis, keepdims=True)

class MiniGPT:
    def __init__(self, vocab=16, d=32, n_heads=4, n_layers=2, seed=0):
        rng = np.random.default_rng(seed)
        s = 0.1
        self.E = rng.normal(size=(vocab, d)) * 0.5
        self.dh = d // n_heads
        self.H, self.D, self.NL = n_heads, d, n_layers
        self.layers = [{
            'Wq': rng.normal(size=(d, d))*s, 'Wk': rng.normal(size=(d, d))*s,
            'Wv': rng.normal(size=(d, d))*s, 'Wo': rng.normal(size=(d, d))*s,
            'W1': rng.normal(size=(d, 4*d))*s*0.8, 'W2': rng.normal(size=(4*d, d))*s*0.8,
            'W3': rng.normal(size=(d, 4*d))*s*0.8,
        } for _ in range(n_layers)]
        self.head = rng.normal(size=(d, vocab)) * s

    def rms(self, x, eps=1e-5):
        return x / np.sqrt((x**2).mean(-1, keepdims=True) + eps)

    def rope(self, x, pos):
        half = x.shape[-1] // 2
        ang = pos[:, None] / (10000 ** (np.arange(half)*2/x.shape[-1]))
        c, s = np.cos(ang), np.sin(ang)
        x1, x2 = x[..., :half], x[..., half:]
        return np.concatenate([x1*c - x2*s, x1*s + x2*c], -1)

    def _attn(self, x, pos, W, cache):
        n = x.shape[0]
        q, k, v = self.rope(x@W['Wq'], pos), self.rope(x@W['Wk'], pos), x@W['Wv']
        if cache is not None:
            k, v = np.concatenate([cache[0], k]), np.concatenate([cache[1], v])
        T = k.shape[0]
        mask = np.triu(np.ones((n, T), bool), T - n + 1)
        outs = []
        for h in range(self.H):
            sl = slice(h*self.dh, (h+1)*self.dh)
            s = q[:, sl] @ k[:, sl].T / math.sqrt(self.dh)
            s = np.where(mask, -np.inf, s)
            outs.append(softmax(s) @ v[:, sl])
        return np.concatenate(outs, -1) @ W['Wo'], (k, v)

    def _ffn(self, x, W):
        h = x @ W['W1']
        return (h / (1 + np.exp(-h)) * (x @ W['W3'])) @ W['W2']

    def forward(self, ids, cache=None):
        pos0 = cache[0][0].shape[0] if cache else 0
        x = self.E[list(ids)]
        new_cache = []
        for i, W in enumerate(self.layers):
            a, kv = self._attn(x, np.arange(pos0, pos0+len(ids)), W,
                               cache[i] if cache else None)
            x = x + a
            x = x + self._ffn(self.rms(x), W)
            new_cache.append(kv)
        logits = self.rms(x) @ self.head
        assert np.isfinite(logits).all(), 'NaN! 查掩码/精度/ε'
        return logits, new_cache

    def generate(self, ids, n, T=1.0, top_p=1.0, rng=None):
        rng = rng or np.random.default_rng()
        logits, cache = self.forward(ids)
        out = []
        for _ in range(n):
            p = softmax(logits[-1] / T)
            if top_p < 1.0:
                order = np.argsort(-p); cum = np.cumsum(p[order])
                keep = order[:np.searchsorted(cum, top_p)+1]
                m = np.zeros_like(p); m[keep] = p[keep]; p = m/m.sum()
            nxt = int(rng.choice(len(p), p=p/p.sum()))
            out.append(nxt)
            logits, cache = self.forward([nxt], cache)
        return list(ids) + out

gpt = MiniGPT()
print('参数量:', sum(v.size for d_ in gpt.layers for v in d_.values()) + gpt.E.size + gpt.head.size)

## 生成(讲义 §3 数字现场)

In [ ]:
ids = [1, 2, 3]
lg, cache = gpt.forward(ids)
gen = list(ids)
for _ in range(8):                                # greedy
    nxt = int(np.argmax(lg[-1]))
    gen.append(nxt)
    lg, cache = gpt.forward([nxt], cache)
print('greedy:', gen)
print('T=0.8 :', gpt.generate([1, 2, 3], 8, T=0.8, rng=np.random.default_rng(7)))

## 出厂检验(讲义 §4)

In [ ]:
# 1) 缓存等价
seq = list(range(1, 12))
lg_full, _ = gpt.forward(seq)
cached = []
lg2, kv2 = gpt.forward(seq[:4])
for t in seq[4:]:
    lg2, kv2 = gpt.forward([t], kv2)
    cached.append(lg2[-1])
err = float(np.abs(np.array(cached) - lg_full[4:]).max())
print('1) 逐 token logits 最大误差:', err)

# 2) isfinite 内建断言:去掉下面注释亲手触发一次(掩码差一)
# bad = np.triu(np.ones((3,3), bool), 0)
# print(bad.astype(int), '← k=0 会把对角线也遮掉')

# 3) 可复现
a = gpt.generate([1,2,3], 8, T=0.8, rng=np.random.default_rng(7))
b = gpt.generate([1,2,3], 8, T=0.8, rng=np.random.default_rng(7))
print('3) 同 seed 重放一致:', a == b)

# 4) 形状不变
lg, _ = gpt.forward(list(range(10)))
print('4) 前向 shape:', lg.shape)

In [ ]:
# 未训练 loss(第11课现场)
seq = list(range(1, 9))
lg, _ = gpt.forward(seq[:-1])
xm = lg.max(-1, keepdims=True)
lp = lg - xm - np.log(np.exp(lg - xm).sum(-1, keepdims=True))
loss = -lp[np.arange(7), seq[1:]].mean()
print('未训练 next-token loss:', round(float(loss), 4), ' 瞎猜基线 ln(16):', round(math.log(16), 4))

In [ ]:
# TODO(C1): 换 seed=1 重建权重,生成是否完全不同?为什么?
# TODO(D3): 计时:200 token,cache on/off
# TODO(D2): 接上第 12/13 课的梯度与 AdamW,训 1000 步看 loss(大工程,加油)